# Day 1 — Data Exploration

**Goal:** Load the TMDB 5000 dataset, understand its structure, and clean the messy columns so we can build the recommender on Day 2.

**What you'll learn here:**
- Loading CSVs with pandas
- Parsing JSON-encoded columns (a real-world data wrangling skill)
- Handling missing values
- Saving a cleaned dataset for downstream use

In [ ]:
import pandas as pd
import numpy as np
import ast  # for parsing the JSON-like strings in some columns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 1. Load the data

In [ ]:
movies = pd.read_csv('../data/tmdb_5000_movies.csv')
credits = pd.read_csv('../data/tmdb_5000_credits.csv')

print(f'Movies shape: {movies.shape}')
print(f'Credits shape: {credits.shape}')
movies.head(2)

In [ ]:
# What columns do we have?
print('MOVIES columns:')
print(movies.columns.tolist())
print('\nCREDITS columns:')
print(credits.columns.tolist())

## 2. Merge the two tables

`credits` has cast & crew info we'll want later. Both tables share `id` / `movie_id`.

In [ ]:
credits = credits.rename(columns={'movie_id': 'id'})
df = movies.merge(credits[['id', 'cast', 'crew']], on='id', how='left')
print(f'Merged shape: {df.shape}')

## 3. Inspect the messy columns

Notice that `genres`, `keywords`, `cast`, `crew` look like JSON but are stored as STRINGS. We need to parse them.

In [ ]:
# Look at one example
print('Type:', type(df['genres'].iloc[0]))
print('Value:', df['genres'].iloc[0][:200], '...')

In [ ]:
def parse_names(json_str):
    """Convert a JSON-like string into a list of 'name' values.
    Example input:  '[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}]'
    Example output: ['Action', 'Adventure']
    """
    if pd.isna(json_str):
        return []
    try:
        parsed = ast.literal_eval(json_str)
        return [item['name'] for item in parsed]
    except (ValueError, SyntaxError):
        return []

df['genres'] = df['genres'].apply(parse_names)
df['keywords'] = df['keywords'].apply(parse_names)
df['production_companies'] = df['production_companies'].apply(parse_names)

df[['title', 'genres', 'keywords']].head(3)

In [ ]:
# Top 3 cast members per movie is usually enough
def parse_top_cast(json_str, n=3):
    if pd.isna(json_str):
        return []
    try:
        parsed = ast.literal_eval(json_str)
        return [item['name'] for item in parsed[:n]]
    except (ValueError, SyntaxError):
        return []

def parse_director(json_str):
    if pd.isna(json_str):
        return None
    try:
        parsed = ast.literal_eval(json_str)
        for member in parsed:
            if member.get('job') == 'Director':
                return member['name']
    except (ValueError, SyntaxError):
        return None
    return None

df['cast'] = df['cast'].apply(parse_top_cast)
df['director'] = df['crew'].apply(parse_director)

df[['title', 'director', 'cast']].head(3)

## 4. Check for missing data

In [ ]:
df[['title', 'overview', 'genres', 'keywords', 'director']].isna().sum()

In [ ]:
# Drop movies with no overview — without one we can't recommend on content
before = len(df)
df = df.dropna(subset=['overview']).reset_index(drop=True)
print(f'Dropped {before - len(df)} movies with no overview. Remaining: {len(df)}')

## 5. Quick sanity check — explore the dataset

In [ ]:
# Most common genres
from collections import Counter
all_genres = [g for genres in df['genres'] for g in genres]
Counter(all_genres).most_common(10)

In [ ]:
# Top 10 highest-rated movies (with at least 1000 votes to filter noise)
df[df['vote_count'] >= 1000].nlargest(10, 'vote_average')[['title', 'vote_average', 'vote_count', 'director']]

## 6. Save the cleaned dataset

We'll save as a pickle so the parsed lists stay as lists (CSVs would re-stringify them).

In [ ]:
# Keep only the columns we'll actually use downstream
clean = df[[
    'id', 'title', 'overview', 'genres', 'keywords',
    'cast', 'director', 'vote_average', 'vote_count',
    'release_date', 'runtime', 'tagline'
]].copy()

clean.to_pickle('../data/movies_clean.pkl')
print(f'Saved {len(clean)} movies to data/movies_clean.pkl')
clean.head(2)

## Day 1 — DONE ✅

**What we have:** a clean dataset of ~4800 movies with parsed genres, keywords, cast, director, ratings.

**Tomorrow (Day 2):** build the TF-IDF content-based recommender on top of this.